# Практика · ML Handbook гл. 40 (sequences)

Источник: [sequences.html](https://mlhandbook-production.up.railway.app/parts/deep-learning/sequences.html)

Теория: [../d.5.40 Последовательности RNN LSTM GRU Transformer.md](../d.5.40 Последовательности RNN LSTM GRU Transformer.md) · [README.md](./README.md)

Deep-ML (опционально): [deep_ml/](./deep_ml/README.md).

**Handbook-код — в этом файле.** Эталон: [handbook_gl40_практика_решения.ipynb](./handbook_gl40_практика_решения.ipynb) **после** попытки.

## Зачем практику handbook

Закрепить главу **руками**: перед каждым блоком — **Зачем этот фрагмент** и ссылка на конспект; после запуска — **Что проверить** по `print`.

| ☐ | Блок |
|---|------|
| ☐ | §40.1 RNN с нуля |
| ☐ | §40.1 nn.RNN + batch_first |
| ☐ | §40.3–40.4 GRU / LSTM |
| ☐ | §40.6.1 attention |
| ☐ | §40.6.2 MultiheadAttention |
| ☐ | §40.6.4 TransformerEncoderLayer |


## Импорты

Один раз в начале.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# §40.1 · RNN-ячейка с нуля

## Зачем этот фрагмент

Руками повторяешь одну RNN-ячейку: те же веса на **каждом** шаге времени и поток h_t. Так проще понять `nn.RNN`, чем сразу с API.

См. [d.5.40 §2.1](../d.5.40 Последовательности RNN LSTM GRU Transformer.md#21-одна-ячейка).

## Что проверить

`states` формы **(6, 8)** — по одному h на каждый из 6 шагов, hidden=8.


In [ ]:
def rnn_cell(x_seq, Wx, Wh, b):
    h = torch.zeros(Wh.shape[0])
    states = []
    for x_t in x_seq:
        h = torch.tanh(Wx @ x_t + Wh @ h + b)
        states.append(h)
    return torch.stack(states)

d_in, d_hidden = 4, 8
Wx = torch.randn(d_hidden, d_in) * 0.1
Wh = torch.randn(d_hidden, d_hidden) * 0.1
b = torch.zeros(d_hidden)
seq = torch.randn(6, d_in)
states = rnn_cell(seq, Wx, Wh, b)
print('скрытых состояний:', tuple(states.shape))


---


# §40.1 · nn.RNN и batch_first

## Зачем этот фрагмент

Сверяешь размеры с документацией PyTorch: где лежит батч, где длина последовательности. На собесе и в дебаге чаще всего путают `batch_first`.

См. [d.5.40 §2](../d.5.40 Последовательности RNN LSTM GRU Transformer.md#2-rnn--память-по-шагам-handbook-401).

Дополнительно: [deep_ml](./deep_ml/README.md).

## Что проверить

`out` **(4, 10, 16)** — h на **каждом** шаге; `h` **(1, 4, 16)** — только финальная память.


In [ ]:
rnn = nn.RNN(input_size=8, hidden_size=16, batch_first=True)
batch = torch.randn(4, 10, 8)
out, h = rnn(batch)
print('выход на каждом шаге:', tuple(out.shape))
print('финальная память h:', tuple(h.shape))


---


# §40.3–40.4 · GRU и LSTM

## Зачем этот фрагмент

GRU хранит одно состояние; LSTM — пару **(h, c)** и отдельные ворота. Сравнение параметров напоминает trade-off «проще / длиннее память» из теории.

См. [d.5.40 §4–5](../d.5.40 Последовательности RNN LSTM GRU Transformer.md#4-gru--две-двери-handbook-403).

## Что проверить

У LSTM `num_layers=2` → первая ось h/c = 2; у GRU out и h совпадают по форме батча.


In [ ]:
batch = torch.randn(4, 10, 8)
gru = nn.GRU(input_size=8, hidden_size=16, batch_first=True)
out_g, h_g = gru(batch)
lstm = nn.LSTM(input_size=8, hidden_size=16, num_layers=2, batch_first=True)
out_l, (h_l, c_l) = lstm(batch)
print('GRU out:', tuple(out_g.shape), 'h:', tuple(h_g.shape))
print('LSTM h:', tuple(h_l.shape), 'c:', tuple(c_l.shape))
print('параметров GRU:', sum(p.numel() for p in gru.parameters()),
      '| LSTM:', sum(p.numel() for p in lstm.parameters()))


---


# §40.6.1 · Scaled dot-product attention

## Зачем этот фрагмент

Без рекуррентности: каждый токен смотрит на все сразу через QKᵀ и softmax. Деление на √d_k — не «украшение», а стабильный градиент.

См. [d.5.40 §7.1](../d.5.40 Последовательности RNN LSTM GRU Transformer.md#71-self-attention-handbook-4061).

Дополнительно: [Deep-ML #59](./deep_ml/README.md).

## Что проверить

`attn` квадратной формы; **сумма весов по строке ≈ 1**.


In [ ]:
def attention(Q, K, V):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / d_k ** 0.5
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

seq_len, d = 4, 8
x = torch.randn(seq_len, d)
Wq, Wk, Wv = (torch.randn(d, d) for _ in range(3))
out, attn = attention(x @ Wq, x @ Wk, x @ Wv)
print('out:', tuple(out.shape), 'attn:', tuple(attn.shape))
print('сумма весов 1-й строки:', round(attn[0].sum().item(), 2))


---


# §40.6.2 · MultiheadAttention

## Зачем этот фрагмент

Несколько голов = несколько «типов связей» параллельно; `embed_dim` обязан делиться на `num_heads`.

См. [d.5.40 §7.2](../d.5.40 Последовательности RNN LSTM GRU Transformer.md#72-multi-head-attention-handbook-4062).

Дополнительно: [Deep-ML #62](./deep_ml/README.md).

## Что проверить

Выход той же формы, что вход: **(1, 5, 16)**.


In [ ]:
mha = nn.MultiheadAttention(embed_dim=16, num_heads=4, batch_first=True)
x = torch.randn(1, 5, 16)
out, _ = mha(x, x, x)
print('mha out:', tuple(out.shape))


---


# §40.6.4 · TransformerEncoderLayer

## Зачем этот фрагмент

Готовый encoder-блок handbook: MHA + FFN + residual + LayerNorm. Форма последовательности **не меняется** — удобно стыковать стеком.

См. [d.5.40 §7.4](../d.5.40 Последовательности RNN LSTM GRU Transformer.md#74-блок-трансформера-encoder-и-decoder-handbook-4064).

Дополнительно: [Deep-ML #88](./deep_ml/README.md).

## Что проверить

`block(seq)` → **(2, 10, 32)**.


In [ ]:
block = nn.TransformerEncoderLayer(d_model=32, nhead=4, dim_feedforward=64, batch_first=True)
seq = torch.randn(2, 10, 32)
print('block out:', tuple(block(seq).shape))
